# Diabetic Retinopathy — Training Notebook (Colab)

**Workflow:**
1. Mount Google Drive
2. Set `STRATEGY` (and `BACKBONE` if fine-tuning)
3. Run all cells — the notebook trains the model, saves a named checkpoint to Drive, and generates a ready-to-submit ZIP.

**Drive layout expected:**
```
MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/
  data/          ← images + train/val/test CSVs
  src/           ← Python modules (copy of repo src/)
  models/
    custom/      ← one .pth per completed run, e.g. custom_auc0.742_20260416_143022.pth
    fine_tune/   ← e.g. fine_tune_alexnet_auc0.821_20260416_150311.pth
  results/
    custom_results/      ← one .csv per completed run
    fine_tune_results/   ← one .csv per completed run
    Submissions/         ← codabench_submission.zip (always up to date)
```

**Naming convention:** `{strategy}[_{backbone}]_auc{val_auc}_{YYYYMMDD_HHMMSS}`

## 1. Mount Drive & set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os

DRIVE_BASE   = '/content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy'
DATA_DIR     = os.path.join(DRIVE_BASE, 'data')
SRC_DIR      = os.path.join(DRIVE_BASE, 'src')
MODELS_DIR   = os.path.join(DRIVE_BASE, 'models')
RESULTS_DIR  = os.path.join(DRIVE_BASE, 'results')

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Create output directories
for d in [
    os.path.join(MODELS_DIR,  'custom'),
    os.path.join(MODELS_DIR,  'fine_tune'),
    os.path.join(RESULTS_DIR, 'custom_results'),
    os.path.join(RESULTS_DIR, 'fine_tune_results'),
    os.path.join(RESULTS_DIR, 'Submissions'),
]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted and paths configured.')

## 2. Install dependencies *(run once if needed)*

In [ ]:
# Uncomment on first run in a fresh Colab session
# !pip install -q scikit-image opencv-python-headless scikit-learn

## 3. Imports

In [ ]:
import random
import shutil
from datetime import datetime

import numpy as np
import numpy.random as npr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

from data.dataset    import RetinopathyDataset
from data.transforms import get_train_transforms, get_val_transforms
from model.custom_net    import CustomNet
from model.fine_tune_net import build_fine_tune_model
from training.trainer    import train_model
from training            import config
from evaluation.submission import (
    test_model, save_strategy_results, generate_submission
)

print('All imports OK.')

## 4. Configuration

**Edit this cell** before each training run.

In [ ]:
# ── Strategy ──────────────────────────────────────────────────────────────────
STRATEGY = 'custom'        # 'custom'  →  train CustomNet from scratch
                            # 'fine_tune' →  fine-tune a pretrained backbone

# ── Fine-tune backbone (ignored when STRATEGY='custom') ───────────────────────
BACKBONE = 'alexnet'        # 'alexnet' | 'resnet18' | 'resnet50' | 'vgg16' | 'efficientnet_b0'

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE   = config.batch_size
NUM_EPOCHS   = config.num_epochs
LR           = config.learning_rate
LR_STEP_SIZE = config.lr_step_size
LR_GAMMA     = config.lr_gamma
MOMENTUM     = config.momentum

# Set to a small integer (e.g. 200) for fast debug runs; 0 = full dataset
MAX_TRAIN_SIZE = 0

# ── Temp checkpoint (overwritten during training, deleted after run) ──────────
TEMP_SAVE_PATH = os.path.join(MODELS_DIR, STRATEGY, '_temp_best.pth')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')
print(f'Strategy : {STRATEGY}')
if STRATEGY == 'fine_tune':
    print(f'Backbone : {BACKBONE}')
print(f'Epochs   : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}')

## 5. Data loading

In [ ]:
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

train_transform = get_train_transforms(img_size=config.img_height)
val_transform   = get_val_transforms(img_size=config.img_height)

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'train.csv'),
    root_dir=DATA_DIR,
    transform=train_transform,
    maxSize=MAX_TRAIN_SIZE,
)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'val.csv'),
    root_dir=DATA_DIR,
    transform=val_transform,
)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'test.csv'),
    root_dir=DATA_DIR,
    transform=val_transform,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256,         shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256,         shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

## 6. Build model

In [ ]:
if STRATEGY == 'custom':
    model = CustomNet().to(device)
else:
    model = build_fine_tune_model(backbone=BACKBONE).to(device)

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
scheduler = lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

## 7. Train

The running best is kept at `_temp_best.pth` during training.  
Once done, it is renamed to a permanent file: `{strategy}[_{backbone}]_auc{auc}_{timestamp}.pth`.

In [ ]:
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

dataloaders   = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

model, best_auc = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    dataloaders=dataloaders,
    dataset_sizes=dataset_sizes,
    device=device,
    num_epochs=NUM_EPOCHS,
    temp_save_path=TEMP_SAVE_PATH,
)

# ── Build permanent name and move temp checkpoint ────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backbone_tag = f'_{BACKBONE}' if STRATEGY == 'fine_tune' else ''
RUN_NAME = f'{STRATEGY}{backbone_tag}_auc{best_auc:.4f}_{timestamp}'

final_model_path = os.path.join(MODELS_DIR, STRATEGY, f'{RUN_NAME}.pth')
if os.path.exists(TEMP_SAVE_PATH):
    shutil.move(TEMP_SAVE_PATH, final_model_path)
    print(f'Model saved -> {final_model_path}')
else:
    # No improvement at all during training — save current weights anyway
    torch.save(model.state_dict(), final_model_path)
    print(f'Model saved (no improvement during training) -> {final_model_path}')

print(f'Run name: {RUN_NAME}')

## 8. Generate submission

- Runs inference on the test set.
- Saves scores as `{run_name}.csv` in `results/{strategy}_results/`.
- Looks for the other strategy's most recent CSV; uses **random scores** if none found.
- Creates `results/Submissions/codabench_submission.zip`.

In [ ]:
# Run test inference
outputs = test_model(model, test_loader, device)
print(f'Test outputs shape: {outputs.shape}')   # expected (1000, 1)

# Save this run's scores with the same descriptive name as the model
current_csv_path = save_strategy_results(outputs, STRATEGY, RESULTS_DIR, RUN_NAME)

# Build submission ZIP (combines both strategies)
zip_path = generate_submission(
    current_strategy=STRATEGY,
    results_dir=RESULTS_DIR,
    current_csv_path=current_csv_path,
    n_test=len(test_dataset),
)
print(f'\nDone! Upload to Codabench: {zip_path}')